# 2D Dataset for Support Vector Machine (SVM)

This notebook generates synthetic two-dimensional datasets, visualizes them, and trains SVM classifiers. It includes:

- A mostly linearly separable dataset for a **linear SVM**.
- A nonlinear `make_moons` dataset for an **RBF-kernel SVM**.
- Train/test evaluation and decision-boundary plots.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

## 2. Helper functions

Each point has two features (`x1`, `x2`) and a binary label (`0` or `1`).

In [ ]:
def plot_dataset(X, y, title):
    plt.figure(figsize=(7, 5))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", alpha=0.8)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.show()


def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    predictions = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, predictions, alpha=0.2, cmap="coolwarm")
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", alpha=0.8)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.show()

## 3. Generate a mostly linearly separable dataset

In [ ]:
X, y = make_classification(
    n_samples=200,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.5,
    random_state=42
)

df = pd.DataFrame(X, columns=["x1", "x2"])
df["label"] = y

print(f"Dataset shape: {df.shape}")
display(df.head())
plot_dataset(X, y, "2D Classification Dataset")

Run the next cell if you want to export the generated data as a CSV file.

In [ ]:
df.to_csv("svm_2d_dataset.csv", index=False)
print("Saved svm_2d_dataset.csv")

## 4. Train and evaluate a linear SVM

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

linear_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear", C=1.0, random_state=42)
)
linear_svm.fit(X_train, y_train)

y_pred = linear_svm.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

plot_decision_boundary(linear_svm, X, y, "Linear SVM Decision Boundary")

## 5. Try a nonlinear dataset with an RBF SVM

The `make_moons` dataset cannot be separated well by a straight line. An RBF kernel allows the SVM to learn a curved decision boundary.

In [ ]:
X_moons, y_moons = make_moons(
    n_samples=200,
    noise=0.15,
    random_state=42
)

plot_dataset(X_moons, y_moons, "Nonlinear 2D Dataset: Two Moons")

X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=42, stratify=y_moons
)

rbf_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
)
rbf_svm.fit(X_train, y_train)

y_pred = rbf_svm.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred))

plot_decision_boundary(rbf_svm, X_moons, y_moons, "RBF SVM Decision Boundary")

## 6. Experiment

Try changing these parameters and observe the effect:

- `noise`: controls overlap in the nonlinear dataset.
- `C`: controls the trade-off between margin size and classification errors.
- `gamma`: controls the influence of individual points for the RBF kernel.
- `class_sep`: controls separation in the first dataset.